[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/03_Training_Strategies/04_multimodal_alignment/04_multimodal_alignment.ipynb)

# 04. Multimodal Alignment Theory

**This notebook covers:**
- Projection head implementation
- Temperature scaling analysis
- CLIP-style alignment training loop
- Embedding space visualization (PCA/t-SNE)

**Runtime:** ~10–15 minutes on CPU

---

> **Theory & derivations:** See [README.md](./README.md) for full step-by-step math.


In [ ]:
# ============================================================
#  Google Colab Setup — Run this cell FIRST
# ============================================================
import os, sys

try:
    import google.colab
    IN_COLAB = True
    print("Google Colab detected — setting up environment...")
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        print("Cloning repository...")
        !git clone --depth 1 {REPO_URL} {REPO_DIR}
    else:
        print("Repository already cloned")

    print("Installing dependencies...")
    !pip install -q -r {REPO_DIR}/requirements.txt

    MODULE_DIR = f"{REPO_DIR}/03_Training_Strategies/04_multimodal_alignment"
    os.chdir(MODULE_DIR)
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    print(f"Colab setup complete — {os.getcwd()}")

    import torch
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("Device: CPU (all notebooks work fine on CPU)")
else:
    os.makedirs("../../assets", exist_ok=True)
    print("Running locally — all set!")

In [ ]:
import sys
sys.path.append('../..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

try:
    from utils.visualization import set_style
    from utils.helpers import count_parameters, get_device
    set_style()
except ImportError:
    def set_style():
        plt.rcParams.update({'figure.figsize': (10, 6), 'figure.dpi': 100})
    def count_parameters(model):
        total = sum(p.numel() for p in model.parameters())
        print(f"Total parameters: {total:,}")
        return total
    def get_device():
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    set_style()

torch.manual_seed(42)
np.random.seed(42)
device = get_device() if callable(get_device) else torch.device('cpu')
print(f"PyTorch {torch.__version__} | Device: {device}")

## 1. Projection Heads

Encoders output $h \in \mathbb{R}^D$; projection maps to contrastive space $\mathbb{R}^d$.


In [ ]:
class ProjectionHead(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=None):
        super().__init__()
        hidden = hidden or in_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(), nn.Linear(hidden, out_dim)
        )
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)

proj = ProjectionHead(256, 128)
h = torch.randn(8, 256)
z = proj(h)
print(f"Projected embeddings shape {z.shape}, norms {z.norm(dim=-1)[:3].tolist()}")

## 2. Temperature Scaling Analysis


In [ ]:
def infonce_loss(sim, tau=0.07):
    logits = sim / tau
    labels = torch.arange(sim.size(0))
    return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2

B = 8
sim = torch.randn(B, B)
sim.fill_diagonal_(2.0)

taus = [0.01, 0.05, 0.07, 0.2, 0.5, 1.0]
losses = [infonce_loss(sim, t).item() for t in taus]

plt.figure(figsize=(8, 4))
plt.plot(taus, losses, 'o-')
plt.xscale('log'); plt.xlabel('Temperature τ'); plt.ylabel('InfoNCE loss')
plt.title('Loss vs Temperature (fixed similarity matrix)')
plt.grid(True, alpha=0.3); plt.show()

## 3. Mini CLIP Alignment Training Loop


In [ ]:
class MiniEncoder(nn.Module):
    def __init__(self, in_dim, embed_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, embed_dim), nn.ReLU(), nn.Linear(embed_dim, embed_dim))
    def forward(self, x):
        return self.net(x)

class MiniCLIP(nn.Module):
    def __init__(self, dim=64, proj_dim=32):
        super().__init__()
        self.image_enc = MiniEncoder(dim, dim)
        self.text_enc = MiniEncoder(dim, dim)
        self.image_proj = ProjectionHead(dim, proj_dim)
        self.text_proj = ProjectionHead(dim, proj_dim)
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1/0.07))

    def forward(self, images, texts):
        vi = self.image_proj(self.image_enc(images))
        vt = self.text_proj(self.text_enc(texts))
        scale = self.logit_scale.exp().clamp(max=100)
        return vi @ vt.T * scale, vi, vt

model = MiniCLIP()
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

# Synthetic paired data: diagonal pairs are positives
N, D = 32, 64
for step in range(100):
    imgs = torch.randn(N, D)
    txts = imgs + 0.3 * torch.randn(N, D)  # noisy paired text features
    logits, _, _ = model(imgs, txts)
    loss = infonce_loss(logits, tau=1.0)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 25 == 0:
        acc = (logits.argmax(1) == torch.arange(N)).float().mean()
        print(f"Step {step:3d} loss={loss.item():.3f} batch-acc={acc.item():.2f}")

## 4. Recall@K and Embedding Visualization


In [ ]:
def recall_at_k(sim, ks=(1, 5)):
    n = sim.size(0)
    ranks = sim.argsort(dim=1, descending=True)
    gt = torch.arange(n).unsqueeze(1)
    return {k: (ranks[:, :k] == gt).any(dim=1).float().mean().item() for k in ks}

with torch.no_grad():
    imgs = torch.randn(N, D)
    txts = imgs + 0.05 * torch.randn(N, D)
    logits, vi, vt = model(imgs, txts)
    sim = vi @ vt.T
    print("Recall@K:", recall_at_k(sim))

try:
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE
    X = torch.cat([vi, vt], dim=0).numpy()
    labels = ['image'] * N + ['text'] * N
    xy = TSNE(n_components=2, perplexity=10, random_state=42).fit_transform(X)
except ImportError:
    xy = PCA(n_components=2).fit_transform(torch.cat([vi, vt], dim=0).numpy())
    labels = ['image'] * N + ['text'] * N

colors = ['tab:blue' if l == 'image' else 'tab:orange' for l in labels]
plt.figure(figsize=(7, 6))
plt.scatter(xy[:, 0], xy[:, 1], c=colors, alpha=0.7, s=30)
for i in range(min(N, 8)):
    plt.plot([xy[i, 0], xy[i+N, 0]], [xy[i, 1], xy[i+N, 1]], 'k-', alpha=0.2)
plt.title('Aligned Embedding Space (pairs connected)'); plt.show()

## Summary

Trained a mini CLIP with projection heads, analyzed temperature, and visualized alignment.

**Next:** [05_scaling_laws](../05_scaling_laws/05_scaling_laws.ipynb)
